# Roteiro das 6 etapas para estudo de caso
## CAP-423 – Ciência de Dados Geoespaciais

Referência para responder questões em que o professor apresenta uma situação-problema e pede métodos e técnicas para resolvê-la.

Para cada etapa: **o que procurar no enunciado** e **qual técnica usar em cada situação**.

> *"Geospatial Data Science is not Data Science plus coordinates. Space must contribute relationships, structure, and geographic meaning to the analysis."* (aula 01)

**Checklist rápido antes de escrever:**

1. Sistema de coordenadas
2. Validação
3. Suporte e escala
4. O que é específico deste lugar?
5. Os números têm comparação?

## Visão geral

| Etapa | Pergunta central | O que a resposta precisa ter |
|---|---|---|
| **1. Diagnóstico** | O que está errado ou arriscado no que foi feito? | Lista de problemas + qual princípio espacial cada um viola |
| **2. Representação e dados** | Os dados medem o que o objetivo pede, na unidade, no lugar e no tempo certos? | Dados escolhidos, suporte, sistema de coordenadas, resolução, período |
| **3. Análise exploratória espacial** | Onde estão as amostras, o que elas cobrem, existe dependência espacial e em que distância? | Cada análise + **a decisão que ela orienta** |
| **4. Modelagem** | Qual modelo usar e como o espaço entra nele? | Método + hipótese sobre o fenômeno + variáveis espaciais |
| **5. Validação** | O teste imita o uso real do modelo? | Forma de dividir os dados + métricas adequadas |
| **6. Interpretação e limitações** | O que o resultado significa e onde não confiar nele? | Incerteza, área de aplicabilidade, limitações, próximos passos |

## 1. Diagnóstico

**O que fazer:** ler o enunciado procurando as "palavras-gatilho" e ligar cada problema a um princípio violado: **dependência espacial**, **heterogeneidade espacial** ou **escala/suporte**.

| Se o enunciado diz… | Problema | Princípio violado |
|---|---|---|
| "amostras concentradas", "perto de estradas/cidades", "coletadas onde era fácil" | Viés amostral; extrapolação | Representatividade |
| "divisão aleatória", "80/20", "10 folds" | Vazamento espacial | Dependência espacial |
| "acurácia de 9X%", "R² = 0,8X" | Resultado otimista; métrica errada para classe rara | Dependência espacial / validação |
| "EPSG:4326 ou 4674" junto com área, distância ou buffer | Medida em graus | Sistema de coordenadas |
| "uma imagem", "uma data" | Falta a dimensão temporal (fenologia) | Representação |
| "focos", "casos", "pontos contados" | O dado não mede o que o objetivo pede | Suporte |
| "resoluções diferentes", "1 km com 10 m" | Mistura de escalas; detalhe falso no mapa | Escala/suporte |
| "dados de anos diferentes" | Descompasso temporal | Representação |
| "várias medidas no mesmo local", "duplicados" | Amostras não são independentes | Dependência |
| "para todo o estado/país", "para o próximo ano" | Domínio de predição maior que o da amostra | Extrapolação |
| "X causa Y", "em toda a região" | Causalidade não provada; efeito suposto igual em todo lugar | Heterogeneidade |
| "latitude/longitude como variáveis" ou "como mais importantes" | Coordenada funcionando como substituta de algo não medido | Modelo não espacialmente explícito |
| "conferido visualmente", "pontos sorteados dentro da classe" | Validação incompleta ou não independente | Validação |

**Também procure:**

- **O que é específico do lugar:** nuvens na Amazônia, cheia no Pantanal, sazonalidade no Cerrado, NDVI saturado em floresta densa.
- **Números sem comparação:** um valor só tem sentido contra uma referência.

## 2. Representação e dados

**O que olhar:**

- O **tipo de Y**: contínuo (regressão) ou categórico (classificação).
- O **suporte** de cada dado: ponto, pixel de quantos metros, polígono.
- O **sistema de coordenadas** de cada camada.
- As **datas** de cada dado.
- A **qualidade**: nuvens, valores faltantes, duplicados.

| Situação | Técnica | Por quê |
|---|---|---|
| Calcular **área** | Reprojetar para **Albers (EPSG:10857)** | Projeção de área igual |
| **Distância ou buffer** numa cidade/região | **UTM da zona** (31981–31985) | Metros com pouca distorção local |
| **Distância** no Brasil todo | **EPSG:5880** | Sistema métrico nacional |
| **Juntar vetor com raster** | Reprojetar o **vetor** | Evita reamostrar o raster |
| Reprojetar **raster de classes** | **Vizinho mais próximo** | Bilinear cria classes que não existem |
| Reprojetar **raster contínuo** | **Bilinear** | Suaviza corretamente |
| Camadas com **resoluções diferentes** | Grade comum (`gdalwarp`); resolução de saída compatível com o objetivo e com os dados | Evita detalhe falso |
| **Uma única data** ou **nuvens** | **Cubo do BDC**, `sits_regularize`, composição por mediana, máscara SCL | Série completa e sem lacunas |
| **Vegetação** em geral | **NDVI** | Vigor da vegetação |
| **Floresta densa** | **EVI** | O NDVI satura |
| **Fogo, corte raso** | **NBR** | Detecta quedas abruptas |
| **Contagens** (casos, focos) | Converter em **taxa** (por população ou área) | Contagem bruta reflete tamanho, não risco |
| **Só dados de presença** | **Pseudo-ausências** com buffer de exclusão | Cria o contraste para o modelo |
| **Duplicados** ou **várias medidas no mesmo local** | Limpar; agregar ou padronizar (mesma profundidade, mesma data) | Cada linha deve ser uma amostra comparável |
| **Datas diferentes** entre Y e X | Compatibilizar os períodos | A covariável tem que descrever o momento da amostra |
| Conferir metadados | `gdalinfo`, `ogrinfo`, `gdf.crs` | Saber o sistema e a resolução de cada camada |

## 3. Análise exploratória espacial

**Regra de ouro:** cada análise precisa terminar em **"→ então eu decido…"**.

| Pergunta | Técnica | Decisão que orienta |
|---|---|---|
| Como Y se distribui? Há classe rara? Faltantes? | Histograma, frequência das classes, contagem de faltantes | Métricas por classe; reforçar a amostragem; limpeza |
| Onde estão as amostras? Estão agrupadas? | **KDE**, quadrats, **Ripley K**, **Clark-Evans** (cuidado com o efeito de borda) | Nova amostragem estratificada; avaliar o viés |
| As amostras cobrem as **condições ambientais** da área? | Comparar as distribuições de X na amostra e na área toda; PCA; **k-means** em X padronizado | Amostrar onde faltam condições; **área de aplicabilidade (AOA)** |
| Quem é vizinho de quem? | Matriz **W**: contiguidade (polígonos), kNN (pontos), distância | Base para Moran, LISA e regressão espacial |
| Y é espacialmente agrupado? | **Moran's I global** com permutação; diagrama de Moran | Esperar vazamento numa divisão aleatória; considerar modelo espacial |
| Onde estão os agrupamentos? | **LISA** (alto-alto, baixo-baixo, outliers) | Hotspots; áreas prioritárias |
| O resultado depende da vizinhança escolhida? | Moran com vários k | Robustez da conclusão |
| **Até que distância** existe dependência? | **Semivariograma** (alcance, pepita, patamar) | **Tamanho dos blocos da validação**; espaçamento das amostras; janelas de vizinhança |
| As variáveis X também são agrupadas? | Moran das variáveis | Entender o que explica o padrão de Y |
| Os valores são muito desiguais? | **Gini**, **Gini espacial** | Desigualdade entre regiões |
| As séries temporais das classes se separam? | `sits_patterns`, dendrograma com **DTW**, **SOM** | Limpar amostras; acrescentar índices |
| Como Y se relaciona com X? | Gráficos de dispersão, correlação | Escolher as variáveis |

## 4. Modelagem

**Sempre diga:** o método + **a hipótese sobre o fenômeno** + **como o espaço entra** no modelo.

| Tipo de problema | Técnica | Hipótese / justificativa |
|---|---|---|
| **Classificar uso e cobertura** com imagens | **Random Forest no sits** sobre séries temporais + limpeza com SOM + cubo de probabilidades + entropia + `sits_smooth` | Classes se separam pelo comportamento ao longo do ano |
| **Mapa contínuo a partir de pontos** (solo, clima) | RF com variáveis ambientais + variáveis espaciais; alternativas: **RFsp**, **RF + krigagem dos resíduos**, **RF-GLS** | Relações não lineares; o espaço entra pelas variáveis ou pelos resíduos |
| **Explicar fatores** em polígonos (bairros, municípios) | Regressão comum → **Moran's I dos resíduos** → modelo espacial se preciso | Inferência sobre os coeficientes |
| O valor dos vizinhos influencia o do local (contágio, difusão) | **Defasagem espacial** (ρWy) | Transbordamento entre vizinhos |
| Causas não medidas seguem um padrão espacial | **Erro espacial** | Variáveis omitidas agrupadas no espaço |
| **O efeito de X muda de região para região** | **GWR** (linear) ou **GRF** (não linear, com importância local) | Heterogeneidade espacial |
| A **proximidade** a algo importa | Variáveis de distância (`sjoin_nearest`), contagem em buffer, interpolação KNN, janelas móveis | O contexto espacial entra como variável |
| **Só presenças** (espécie) | RF de classificação com **pseudo-ausências** | Contraste entre presença e fundo |
| **Sem rótulos**, descobrir padrões | **k-means**, **HCA**, **SOM**, **SOM + HCA** | Agrupar no espaço de atributos |
| Delimitar **objetos** (talhões, cicatrizes, manchas) | **Crescimento de regiões**, **SLIC**, **watershed**; espaço-temporal: **SOM + HCA + filtro majoritário** | A unidade de análise é o objeto, não o pixel |
| Mapa com **ruído "sal e pimenta"** | `sits_smooth` ou filtro majoritário | Vizinhos tendem a ter a mesma classe |
| **Classe rara** | Amostragem balanceada; reforço da classe | Evita que o modelo ignore a classe |
| Coordenadas como variáveis | Usar com cautela, ou trocar por variáveis com significado | Coordenadas generalizam mal e não explicam o fenômeno |

**As 5 estratégias da aula 05**, para citar:

1. ML clássico + variáveis não espaciais + validação aleatória.
2. ML clássico + todas as variáveis espaciais + validação aleatória.
3. ML clássico + todas as variáveis espaciais + validação espacial.
4. ML clássico + variáveis espaciais selecionadas + validação espacial.
5. ML espacial + variáveis selecionadas + validação espacial.

O alvo é chegar à **4 ou à 5**.

## 5. Validação

**Regra de ouro:** *a validação deve reproduzir o cenário em que o modelo será usado.*

| Cenário de uso | Estratégia |
|---|---|
| Prever **dentro** da área já amostrada | Validação cruzada espacial (**blocos** ou **clusters**) |
| Prever **numa região nova** | **Teste espacial independente**: deixar regiões inteiras de fora |
| Prever **o próximo ano** | **Validação temporal**: treinar com anos anteriores e testar no seguinte |
| Prever em **condições ambientais novas** | **kNNDM**; conferir a **AOA**; separar só no espaço pode não bastar |
| **Muitos dados** | Validação espacial para ajustar + **teste independente usado uma vez** |
| **Poucos dados** | Só validação cruzada espacial |
| Dados com dependência espacial | Blocos **maiores que o alcance** do semivariograma, ou **LOO com buffer** |
| Várias medidas do **mesmo local** | Manter todas **no mesmo fold** |

**Métrica por tipo de problema:**

| Problema | Métricas |
|---|---|
| Regressão | **RMSE**, MAE, R² |
| Classificação | **Matriz de confusão**, acurácia global, **acurácias do produtor e do usuário**, **F1**, kappa |
| Probabilidade (presença/ausência) | **AUC ROC** |
| Segmentação | **Precisão, revocação, F, IoU** |
| Modelos de regressão espacial | **AIC** + Moran's I dos resíduos |

**Cuidados com os dados de referência:**

- Amostrar **todas as classes** (aleatória estratificada).
- Referência **independente**: intérprete que não conhece o mapa (interpretação cega), imagem de maior resolução ou produto oficial.
- O teste **nunca** entra no treino nem no ajuste do modelo.

## 6. Interpretação e limitações

| O que avaliar | Técnica | O que concluir |
|---|---|---|
| Quanto o resultado cai da validação aleatória para a espacial | Comparar as duas | A queda mostra quanto do resultado era vazamento; o valor espacial é o honesto |
| **Onde** o modelo erra | Mapa dos erros; **Moran/LISA dos resíduos** | Erros agrupados indicam viés geográfico ou estrutura que falta no modelo |
| Onde o modelo está **inseguro** | **Entropia**, margem (`sits_uncertainty`) | Onde coletar novas amostras (*active learning*) |
| Onde o modelo **não deve ser usado** | **Área de aplicabilidade (AOA)** | Mascarar ou sinalizar as áreas de extrapolação |
| Quais variáveis importam | Importância global; **importância local (GRF)**; coeficientes da GWR | Coordenada muito importante = substituta de algo não medido; o efeito pode variar por região |
| Afirmação de causa | Associação não é causa | Use "está associado a", não "causa" |
| Classes muito confundidas | Matriz de confusão | Mais índices, limpar amostras ou fundir classes (declarando isso) |
| Escala e suporte | Comparar a resolução de entrada com a de saída; testar outra agregação | Detalhe falso; resultado depende da unidade espacial |
| Onde coletar novas amostras | Lacunas no **espaço de atributos** + alta incerteza + fora da AOA (não só onde há poucos pontos no mapa) | Plano da próxima coleta |
| Limitações inerentes | Nuvens, fronteiras difusas, dados antigos, ruído do sensor | Declarar com transparência |

## Anexo: sistemas de coordenadas (cola rápida)

**Graus ou metros? Olhe os números:**

| Se os números são… | Então está em | Exemplo |
|---|---|---|
| Pequenos, com vírgula, entre −180 e 180 | **Graus** | (−47,88 ; −15,79) |
| Grandes, com 6 ou 7 dígitos | **Metros** | (350 000 ; 7 400 000) |

**Códigos EPSG para decorar:**

| Código | Nome | Unidade | Para quê |
|---|---|---|---|
| **4326** | WGS 84 | graus | GPS, dados globais, GeoJSON |
| **4674** | SIRGAS 2000 | graus | padrão oficial brasileiro |
| **31981 a 31985** | SIRGAS 2000 / UTM 21S a 25S | metros | distância e buffer numa região |
| **5880** | SIRGAS 2000 / Policônica | metros | distância no Brasil todo |
| **10857** | SIRGAS 2000 / Brazil Albers | metros | **área** (estatística nacional, BDC) |
| **3857** | Pseudo-Mercator | metros, mas distorcidos | **só para mapas web** |

**Zonas UTM de algumas cidades:** Cuiabá 21S (31981) · Goiânia 22S (31982) · Brasília e São Paulo 23S (31983) · Recife 25S (31985).

**Na definição do sistema:** `GEOGCRS`, `+proj=longlat` ou `unit = degree` → graus. `PROJCRS`, `+proj=utm`, `+proj=aea` ou `unit = metre` → metros. No Python: `gdf.crs.is_geographic` → `True` = graus.

**Roteiro de 3 perguntas** (sempre que houver área, distância, buffer ou sobreposição):

1. Em que sistema estão os dados?
2. O que vai ser medido? Área → Albers. Distância ou buffer → UTM ou outro sistema métrico.
3. Os dados estão no mesmo sistema? Se não, reprojete o vetor.